# The syntax tour — executable

Every cell below RUNS (it executes in CI and under `uv run pytest`, so the
tour can never rot). The exceptions carry the `skip-execution` tag and say
why: either a **committed future** — a ratified spelling whose machinery
arrives with a named milestone — or a **device cell** that needs a WebGPU
adapter (run those locally on the Mac).

Refusals are part of the syntax: cells that demonstrate one catch it and
print the message, because the messages are frozen API.

Normative sources: `docs/design/200_the-spec.md` (the spec),
`250_coordinate-algebra.md` (the indexing law). This tour pops around;
read it in any order.

## Frames, Coordinates, and host indexing (250)

A dim's identity is its observable **Frame**; a **Coordinate** is a point
on one, in-bounds by construction. Subscripts are named, order-free, and
typed — and indexing never promotes to a scalar.

In [1]:
import numpy as np
from pdum.tl import Tensor, extent


def T(a, names):
    return Tensor.from_numpy(np.asarray(a, dtype=np.float64), names)

img = T(np.arange(48).reshape(6, 8), ("y", "x"))
y, x = img.frames("y", "x")          # the host coordinate factories
cell = img[y[2], x[5]]               # order-free; a full point is RANK-0, never a scalar
crop = img[y[1]:y[5], x[0]:x[4]:2]   # Coordinate endpoints; step decimates
print(cell.item(), crop.sizes(), extent(y[2]))   # .item() is the one scalar exit

21.0 {'y': 4, 'x': 2} 6


In [2]:
# The torsor law: points and differences are different sorts.
d = y[5] - y[2]                      # Coordinate - Coordinate -> Displacement
print(y[2] + d, d)                   # Coordinate + Displacement -> Coordinate
try:
    y[2] + y[3]                      # adding points is the affine crime
except TypeError as e:
    print("refused:", e)
try:
    y[2] * 0.5                       # no arithmetic on a point — coerce explicitly
except TypeError as e:
    print("refused:", e)

y[5] y[+3]
refused: cannot add two Coordinates: adding points is the affine crime — subtract for a Displacement, or add a Displacement to a point
refused: no numeric arithmetic on a Coordinate — a point is not a number; coerce explicitly (.i for the lattice int, .phys for the physical reading)


In [3]:
# Stores: scalars promote to a const broadcast over the view (pointwise's law).
img[y[0]:y[2]] = 7.0
print(img.to_numpy()[:3, :3])

[[ 7.  7.  7.]
 [ 7.  7.  7.]
 [16. 17. 18.]]


## The assemblage tier (S.1)

Whole-tensor declarations: pointwise with alignment as the gatekeeper,
`iota` as the label→value bridge, reductions that name what they own.

In [4]:
from pdum.tl import iota, pointwise, pw, red, reduce

t = T(np.arange(12).reshape(3, 4), ("y", "x"))
ramp = iota(t, "x")                            # coordinates as data (closed form)
s = pointwise(pw.add, t, ramp)
print(reduce(red.sum, s, ("x",)).to_numpy())   # the reduction NAMES its dims

[12. 28. 44.]


## Compute kernels (S.3)

`global_idx` is the standard door — my position in the writable, any
launch geometry (it is the merge map over the raw pair; `thread_idx`
stays the primitive for block-local work). Coordinates are typed:
value math goes through `f32`, index math through `i32`, and
`extent(c)` reads the domain width from the coordinate itself.

In [5]:
from pdum.tl import compute, extent, f32, global_idx, i32, thread_idx  # noqa: F401


@compute
def shade(img):
    i, j = global_idx("y", "x")
    u = f32(j) / f32(extent(j))
    v = f32(i) / f32(extent(j))     # one denominator: aspect preserved
    img[i, j] = u + 10.0 * v

out = T(np.zeros((4, 8)), ("y", "x"))
shade(out)
print(out.to_numpy()[:2])

[[0.    0.125 0.25  0.375 0.5   0.625 0.75  0.875]
 [1.25  1.375 1.5   1.625 1.75  1.875 2.    2.125]]


In [6]:
# One spelling, every geometry: the same kernel under an explicit tile launch.
from pdum.tl import config

big = T(np.zeros((8, 16)), ("y", "x"))
shade[config(blocks=(2, 2), threads=(4, 8))](big)
print(np.allclose(big.to_numpy()[:4, :8].shape, (4, 8)))

True


In [7]:
# Buffer reads at computed indices ride i32 (the read door);
# subscripts of Coordinates are order-free and typed.
@compute
def box3(tex, dst):
    (i,) = global_idx("y")
    acc = 0.0
    for di in range(3):
        acc = acc + tex[i32(i) + di]
    dst[i] = acc / 3.0

tex, dst = T(np.arange(5.0), ("y",)), T(np.zeros(3), ("y",))
box3(tex, dst)
print(dst.to_numpy())

[1. 2. 3.]


In [8]:
# rename is the adapter when a tensor's dims don't share the lattice names.
from pdum.tl import rename  # noqa: F401 — resolved from the body's globals


@compute
def copy_across(src, dst):
    (i,) = global_idx("y")
    dst[i] = src[rename(i, "row")] * 2.0

src = T(np.arange(4.0), ("row",))
dst2 = T(np.zeros(4), ("y",))
copy_across(src, dst2)
print(dst2.to_numpy())

[0. 2. 4. 6.]


## Device functions and the call-boundary law

Coordinates cross calls AS Coordinates — the cast site is yours: the
caller casts (`f(f32(i), f32(j))`, f a plain scalar citizen) or the
callee casts (a frame-aware f, which can read `extent` off its
argument). And `fwidth` IS the analytic wrt-ambient derivative —
`value_and_grad` staged in the body, no 2×2 quad.

In [9]:
from pdum.dsl.intrinsics import clamp  # noqa: F401 — inlines by capture-and-call
from pdum.dsl.markers import sqrt  # noqa: F401 — bare in bodies

from pdum.dsl import jit, value_and_grad


def circle(cy, cx, r):
    @jit()
    def go(y, x):
        d = sqrt((y - cy) * (y - cy) + (x - cx) * (x - cx))
        return d - r                        # signed distance
    return go

@compute
def aa_shader(f, img):
    i, j = global_idx("y", "x")
    g = value_and_grad(f, wrt=("y", "x"))   # staged: f's identity is compile-time
    v, (dy, dx) = g(f32(i), f32(j))         # the pattern declares the structure
    w = sqrt(dy * dy + dx * dx)             # fwidth — analytic, no 2×2 quad
    img[i, j] = clamp(v / w + 0.5, 0.0, 1.0)

disk = T(np.zeros((16, 16)), ("y", "x"))
aa_shader(circle(7.5, 7.5, 5.0), disk)
print(np.round(disk.to_numpy()[7], 2))      # a one-pixel analytic AA band

[1.   1.   1.   0.03 0.   0.   0.   0.   0.   0.   0.   0.   0.03 1.
 1.   1.  ]


## Records (surface C, 200 §4)

A frozen dataclass, DECLARED: its constructor joins the vocabulary,
fields fold at lowering, and the derivative type law is per-field —
`d(RG)/dy` IS an RG.

In [10]:
from dataclasses import dataclass

from pdum.dsl.registry import DEFAULT
from pdum.dsl.surfaces import record


@dataclass(frozen=True)
class RG:
    r: float
    g: float

record(DEFAULT, RG)

@jit()
def probe(y, x):
    c = RG(y * x, y + x)
    dc = with_respect_to(c, y)  # noqa: F821 — RG(x, 1): the record clause
    return dc.r + dc.g

@compute
def k(f, img):
    i, j = global_idx("y", "x")
    img[i, j] = f(f32(i), f32(j))

rec_out = T(np.zeros((2, 3)), ("y", "x"))
k(probe, rec_out)
print(rec_out.to_numpy())  # x + 1

[[1. 2. 3.]
 [1. 2. 3.]]


## Graphics (S.4): record vertex buffers, pairing, render

The vertex ambient is `thread_idx("vertex_id")` — the draw domain. A
RECORD vertex buffer's structured dtype is the memory shape: fields by
name, no anonymous columns. Return is mandatory; varyings are claimed
by naming them.

In [11]:
from pdum.tl.graphics import fragment, pair, position, render, vertex

MESH_DT = np.dtype([("px", "<f8"), ("py", "<f8")])
tri = Tensor.from_numpy(
    np.array([(-1.0, -1.0), (1.0, -1.0), (-1.0, 1.0)], dtype=MESH_DT), ("vertex_id",)
)

@vertex
def mesh(verts):
    (vid,) = thread_idx("vertex_id")
    p = verts[vid]                   # the record element: fields by NAME
    u = p.px * 0.5 + 0.5             # a claimed varying  # noqa: F841
    return position(p.px, p.py)

@fragment
def shade_frag(varying):
    return varying.u

target = T(np.zeros((6, 6)), ("y", "x"))
render(pair(mesh, shade_frag), tri, target=target)
print(np.round(target.to_numpy(), 2))

[[0.08 0.25 0.42 0.58 0.75 0.92]
 [0.08 0.25 0.42 0.58 0.75 0.  ]
 [0.08 0.25 0.42 0.   0.   0.  ]
 [0.08 0.25 0.   0.   0.   0.  ]
 [0.08 0.25 0.   0.   0.   0.  ]
 [0.08 0.   0.   0.   0.   0.  ]]


In [12]:
# A record TAP: the fragment binds a DECLARED record of its fields —
# the tap buffer is a struct-element tensor matched by name. A G-buffer is one tap.
@dataclass(frozen=True)
class GBuf:
    lum: float
    depth: float

record(DEFAULT, GBuf)

@vertex
def quad():
    (vid,) = thread_idx("vertex_id")
    i = i32(vid)
    u = 1.0 if (i == 1 or i == 3 or i == 4) else 0.0
    v = 1.0 if (i == 2 or i == 4 or i == 5) else 0.0
    return position(u * 2.0 - 1.0, v * 2.0 - 1.0)

@fragment
def gshade(varying):
    lum = varying.u * 0.5
    g = GBuf(lum, varying.v)  # noqa: F841 — ONE claimed record site
    return lum

gdt = np.dtype([("lum", "<f8"), ("depth", "<f8")])
gbuf = Tensor.from_numpy(np.zeros((4, 4), dtype=gdt), ("y", "x"))
gimg = T(np.zeros((4, 4)), ("y", "x"))
render(pair(quad, gshade)[config(taps={"g": gbuf})], target=gimg)
print(np.round(gbuf.field("depth").to_numpy(), 2))

[[0.12 0.12 0.12 0.12]
 [0.38 0.38 0.38 0.38]
 [0.62 0.62 0.62 0.62]
 [0.88 0.88 0.88 0.88]]


## The render loop today (reference tier)

The zoo's cylinder: a record mesh rippled by a compute kernel over its
field views (a warm phase uniform), the rotation an in-shader uniform,
the analytic-AA fragment. Reference semantics — the device twin lives
in `conformance/`.

In [13]:
from pdum.tl.zoo.cylinder import demo_frames

frames = demo_frames(angles=(0.0, 1.2), size=(24, 32))
print(len(frames), frames[0].shape, round(float(frames[0].std()), 3))

2

 (24, 32) 0.266


## COMMITTED FUTURE — the resident render loop (L2: device-resident state)

The cell below is the ratified residency contract, tagged
`skip-execution` until the machinery lands (200 §8, device-resident
state + the epoch/ownership handshake). The laws it spells: transfers
are EXPLICIT one-time boundary acts; residency is a property of the
tensor's buffer, never of a kernel; the render binds THE SAME buffer
the compute wrote (usage flags are ours) — zero copies in the loop; the
user never names a bind group; readback is an explicit act.

In [ ]:
# a COMMITTED FUTURE (200 §8: device-resident state) — names below arrive with L2
mesh_d = mesh_host.to("webgpu")            # ONE conscious transfer — the boundary act  # noqa: F821
rippled_d = device_like(mesh_d)            # allocated RESIDENT; never touches the host  # noqa: F821
for angle in angles:  # noqa: F821
    ripple(mesh_d.field("theta"), mesh_d.field("h"),  # noqa: F821
           rippled_d.field("theta"), rippled_d.field("h"))     # compute ON the device
    render(pair(spun(angle), shade), rippled_d, g, target=canvas)  # noqa: F821
    # the render binds THE SAME buffer the compute wrote — no copy, no bind-group ceremony
image = canvas.readback()                  # leaving the device is EXPLICIT  # noqa: F821

## DEVICE — the conformance twin (needs a WebGPU adapter; run locally)

Tagged `skip-execution` for CI. The conformance executor is
translation-only and deliberately re-uploads per call — the residency
contract above is the real thing it stands in for.

In [ ]:
import sys

sys.path.insert(0, "../../../conformance")
from wgsl_executor import render_wgpu

got = render_wgpu(pair(mesh, shade_frag), tri, shape=(6, 6))
print(np.round(got, 2))  # pixel-identical to the reference render above